# N$_2$O - binning

Bin the observational network.

## Imports

In [1]:
from pathlib import Path

import openscm_units
import pandas as pd
import pint
from pydoit_nb.config_handling import get_config_for_step_id

import local.binned_data_interpolation
import local.binning
import local.raw_data_processing
from local.config import load_config_from_file

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [2]:
pint.set_application_registry(openscm_units.unit_registry)  # type: ignore

## Define branch this notebook belongs to

In [3]:
step: str = "calculate_n2o_monthly_fifteen_degree_pieces"

## Parameters

In [4]:
config_file: str = "../../dev-config-absolute.yaml"  # config file
step_config_id: str = "only"  # config ID to select for this branch

In [5]:
# Parameters
config_file = "/Users/znicholls/Documents/repos/CMIP-GHG-Concentration-Generation/output-bundles/v1.0.0/v1.0.0-config.yaml"
step_config_id = "only"


## Load config

In [6]:
config = load_config_from_file(Path(config_file))
config_step = get_config_for_step_id(config=config, step=step, step_config_id=step_config_id)

# # Don't use surface flask network as it is already included in HATS
# config_process_noaa_surface_flask_data = get_config_for_step_id(
#     config=config,
#     step="process_noaa_surface_flask_data",
#     step_config_id=config_step.gas,
# )
config_process_noaa_hats_data = get_config_for_step_id(
    config=config,
    step="process_noaa_hats_data",
    step_config_id=config_step.gas,
)
config_process_agage_data_gc_md = get_config_for_step_id(
    config=config,
    step="retrieve_and_extract_agage_data",
    step_config_id=f"{config_step.gas}_gc-md_monthly",
)
config_process_ale_data = get_config_for_step_id(
    config=config, step="retrieve_and_extract_ale_data", step_config_id="monthly"
)
config_process_gage_data = get_config_for_step_id(
    config=config, step="retrieve_and_extract_gage_data", step_config_id="monthly"
)

## Action

### Load data

In [7]:
all_data_l = []
for f, dep_short_names in [
    (
        config_process_noaa_hats_data.processed_monthly_data_with_loc_file,
        local.dependencies.load_source_info_short_names(
            config_process_noaa_hats_data.source_info_short_names_file,
        ),
    ),
    (
        config_process_agage_data_gc_md.processed_monthly_data_with_loc_file,
        local.dependencies.load_source_info_short_names(
            config_process_agage_data_gc_md.source_info_short_names_file
        ),
    ),
    (
        config_process_ale_data.processed_monthly_data_with_loc_file,
        [config_process_ale_data.source_info.short_name],
    ),
    (
        config_process_gage_data.processed_monthly_data_with_loc_file,
        [config_process_gage_data.source_info.short_name],
    ),
]:
    try:
        all_data_l.append(local.raw_data_processing.read_and_check_binning_columns(f))
    except Exception as exc:
        msg = f"Error reading {f}"
        raise ValueError(msg) from exc

    for dsn in dep_short_names:
        local.dependencies.save_dependency_into_db(
            db=config.dependency_db,
            gas=config_step.gas,
            dependency_short_name=dsn,
        )

all_data = pd.concat(all_data_l)
all_data["gas"] = all_data["gas"].str.lower()
all_data = all_data[all_data["gas"] == config_step.gas]
all_data

,year,month,value,site_code,latitude,longitude,gas,source,unit,network,station,measurement_method,time,std. dev.,numb,instrument
0,1988,2,308.582,alt,82.50,-62.30,n2o,hats,ppb,NOAA,alt,hats,NaN,NaN,NaN,NaN
1,1988,3,308.441,alt,82.50,-62.30,n2o,hats,ppb,NOAA,alt,hats,NaN,NaN,NaN,NaN
2,1988,4,308.332,alt,82.50,-62.30,n2o,hats,ppb,NOAA,alt,hats,NaN,NaN,NaN,NaN
3,1988,5,308.259,alt,82.50,-62.30,n2o,hats,ppb,NOAA,alt,hats,NaN,NaN,NaN,NaN
4,1988,6,308.218,alt,82.50,-62.30,n2o,hats,ppb,NOAA,alt,hats,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3908,1991,9,309.126,SMO,-14.23,-170.56,n2o,GAGE,ppb,GAGE,smo,GAGE,NaN,NaN,NaN,NaN
3909,1992,9,310.013,SMO,-14.23,-170.56,n2o,GAGE,ppb,GAGE,smo,GAGE,NaN,NaN,NaN,NaN
3910,1993,9,309.886,SMO,-14.23,-170.56,n2o,GAGE,ppb,GAGE,smo,GAGE,NaN,NaN,NaN,NaN
3911,1994,9,310.220,SMO,-14.23,-170.56,n2o,GAGE,ppb,GAGE,smo,GAGE,NaN,NaN,NaN,NaN


## Bin and average data

- all measurements from a station are first averaged for the month
- then average over all stations
    - stations get equal weight
    - flask/in situ networks (i.e. different measurement methods/techniques)
      are treated as separate stations i.e. get equal weight
- this order is best as you have a better chance of avoiding giving different times more weight by accident
    - properly equally weighting all times in the month would be very hard,
      because you'd need to interpolate to a super fine grid first (one for future research)

In [8]:
all_data_with_bins = local.binning.add_lat_lon_bin_columns(all_data)
all_data_with_bins

,year,month,value,site_code,latitude,longitude,gas,source,unit,network,station,measurement_method,time,std. dev.,numb,instrument,lon_bin,lat_bin
0,1988,2,308.582,alt,82.50,-62.30,n2o,hats,ppb,NOAA,alt,hats,NaN,NaN,NaN,NaN,-90.0,82.5
1,1988,3,308.441,alt,82.50,-62.30,n2o,hats,ppb,NOAA,alt,hats,NaN,NaN,NaN,NaN,-90.0,82.5
2,1988,4,308.332,alt,82.50,-62.30,n2o,hats,ppb,NOAA,alt,hats,NaN,NaN,NaN,NaN,-90.0,82.5
3,1988,5,308.259,alt,82.50,-62.30,n2o,hats,ppb,NOAA,alt,hats,NaN,NaN,NaN,NaN,-90.0,82.5
4,1988,6,308.218,alt,82.50,-62.30,n2o,hats,ppb,NOAA,alt,hats,NaN,NaN,NaN,NaN,-90.0,82.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3908,1991,9,309.126,SMO,-14.23,-170.56,n2o,GAGE,ppb,GAGE,smo,GAGE,NaN,NaN,NaN,NaN,-150.0,-7.5
3909,1992,9,310.013,SMO,-14.23,-170.56,n2o,GAGE,ppb,GAGE,smo,GAGE,NaN,NaN,NaN,NaN,-150.0,-7.5
3910,1993,9,309.886,SMO,-14.23,-170.56,n2o,GAGE,ppb,GAGE,smo,GAGE,NaN,NaN,NaN,NaN,-150.0,-7.5
3911,1994,9,310.220,SMO,-14.23,-170.56,n2o,GAGE,ppb,GAGE,smo,GAGE,NaN,NaN,NaN,NaN,-150.0,-7.5


In [9]:
print(local.binning.get_network_summary(all_data_with_bins))

Collating data from:
- AGAGE gc-md (5 stations: cgo, mhd, rpb, smo, thd)
- ALE ALE (5 stations: adr, cgo, org, rpb, smo)
- GAGE GAGE (5 stations: cgo, mhd, org, rpb, smo)
- NOAA hats (12 stations: alt, brw, cgo, kum ... smo, spo, sum, thd)


In [10]:
bin_averages = local.binning.calculate_bin_averages(all_data_with_bins)
bin_averages

Will ignore columns: ['site_code', 'latitude', 'longitude', 'source', 'time', 'std. dev.', 'numb', 'instrument']
Took mean over ['index']
Took mean over ['measurement_method', 'network', 'station']


,gas,unit,year,month,lat_bin,lon_bin,value
0,n2o,ppb,1977,9,-82.5,30.0,302.041
1,n2o,ppb,1977,9,22.5,-150.0,301.092
2,n2o,ppb,1977,9,37.5,-90.0,300.152
3,n2o,ppb,1977,10,-82.5,30.0,301.695
4,n2o,ppb,1977,10,22.5,-150.0,301.156
...,...,...,...,...,...,...,...
5871,n2o,ppb,2024,1,-82.5,30.0,336.736
5872,n2o,ppb,2024,1,22.5,-150.0,337.939
5873,n2o,ppb,2024,1,37.5,-90.0,337.863
5874,n2o,ppb,2024,1,52.5,-30.0,337.955


### Save

In [11]:
local.binned_data_interpolation.check_data_columns_for_binned_data_interpolation(bin_averages)
assert set(bin_averages["gas"]) == {config_step.gas}

In [12]:
config_step.processed_bin_averages_file.parent.mkdir(exist_ok=True, parents=True)
bin_averages.to_csv(config_step.processed_bin_averages_file, index=False)
bin_averages

,gas,unit,year,month,lat_bin,lon_bin,value
0,n2o,ppb,1977,9,-82.5,30.0,302.041
1,n2o,ppb,1977,9,22.5,-150.0,301.092
2,n2o,ppb,1977,9,37.5,-90.0,300.152
3,n2o,ppb,1977,10,-82.5,30.0,301.695
4,n2o,ppb,1977,10,22.5,-150.0,301.156
...,...,...,...,...,...,...,...
5871,n2o,ppb,2024,1,-82.5,30.0,336.736
5872,n2o,ppb,2024,1,22.5,-150.0,337.939
5873,n2o,ppb,2024,1,37.5,-90.0,337.863
5874,n2o,ppb,2024,1,52.5,-30.0,337.955
